In [3]:
from rds import get_rds_connection, create_table, create_index
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
create_table(conn)
create_index(conn)
conn.close()

Creating search table...
Search table created.
Creating index...
Index created.


In [1]:
from rds import get_rds_connection, backfill
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
backfill(conn)
conn.commit()
conn.close()

Backfilling theorem_search_qwen...
Rows inserted: 0


In [ ]:
from rds import get_rds_connection
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
cur = conn.cursor()

print("Configuring session...")

cur.execute("SET maintenance_work_mem = '8GB';")
cur.execute("SET max_parallel_maintenance_workers = 12;")
cur.execute("SET max_parallel_workers = 16;")
cur.execute("SET synchronous_commit = OFF;")

print("Creating index...")
cur.execute(r"""
CREATE INDEX CONCURRENTLY theorem_embedding_gemma_hnsw
ON theorem_embedding_gemma
USING hnsw (embedding vector_cosine_ops)
WITH (
  m = 16,
  ef_construction = 128
);
""")

print("Success.")

cur.close()
conn.close()

Configuring session...
Creating index...


In [1]:
from rds import get_rds_connection
from dotenv import load_dotenv
load_dotenv()

conn = get_rds_connection()
cur = conn.cursor()

print("Executing query...")
cur.execute(r"""
CREATE TABLE arxiv_umap_sample AS
WITH eligible AS (
  SELECT
    t.theorem_id,
    p.primary_category
  FROM theorem t
  JOIN paper p ON p.paper_id = t.paper_id
  WHERE p.source = 'arXiv'
    AND p.primary_category = ANY(ARRAY[
      'math.AP','math.CO','math.AG','math.PR','math.NT',
      'math.DG','math.DS','math.FA','math.RT','math.GR'
    ])
),
sampled AS (
  SELECT
    theorem_id,
    primary_category,
    ROW_NUMBER() OVER (PARTITION BY primary_category ORDER BY RANDOM()) AS rn
  FROM eligible
)
SELECT
  theorem_id,
  primary_category
FROM sampled
WHERE rn <= 1000;
""")

print("Success.")

cur.close()
conn.close()

Executing query...
Success.
